# E0: reproduce FLOCK zero-shot results

1. In the notebook settings, select the **GPU T4** accelerator and turn **Internet on**.
2. Choose **Save Version**, then **Save & Run All**.
3. Download the `e0_*` files from the Output tab.

The notebook checks the walker, runs three small inductive graphs for comparison with Table 9, and
times 20 Hetionet triples at the paper's zero-shot setting (n = 512, P = 2).

In [ ]:
import base64, io, json, os, subprocess, tarfile
from pathlib import Path

EXPERIMENT_COMMIT = "1dccc522c974e4a5dbb1214fd47830bb0ed78965"
UPSTREAM_COMMIT = "f35103d25a78bdf4075de5c673a51de4979aa4d7"
WORK = Path(os.environ.get("E0_WORK", "/tmp/e0"))
OUTPUT = Path(os.environ.get("E0_OUTPUT", "/kaggle/working"))
REPO = WORK / "flock"
PYTHON = Path(os.environ.get("E0_PYTHON", WORK / "env/bin/python"))
SCRIPTS = REPO / "experiments/e0_reproduction"
RUN_ENV = {
    **os.environ,
    "EXPERIMENT_COMMIT": EXPERIMENT_COMMIT,
    "CUDA_VISIBLE_DEVICES": "0",
    "TQDM_DISABLE": "1",
}
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def sh(command, check=True, cwd=None):
    print("$", command, flush=True)
    bash_command = f"set -o pipefail; {command}"
    code = subprocess.run(bash_command, shell=True, executable="/bin/bash", cwd=cwd, env=RUN_ENV).returncode
    if code and check:
        raise RuntimeError(f"command failed with exit code {code}")
    if code:
        print(f"FAILED with exit code {code}; continuing", flush=True)
    return code == 0

In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout
print(gpu)
assert "T4" in gpu, "Select the GPU T4 accelerator in the notebook settings."
sh("nproc")

In [ ]:
if not REPO.exists():
    sh(f"git clone -q https://github.com/jw9730/flock {REPO}")
sh(f"git -C {REPO} checkout -q {UPSTREAM_COMMIT}")
BUNDLE = "H4sIAAAAAAAAA+19+3PbyLVmfuZfgeKt3aUSioP3wxVurWPLM97YHpfHk9xbuloMREIiRiTBEKBtJXfmb7/ndDeArwFSAmU7yVbEclkE0N2nH6fP6zsNbpJP8fUyv0yW8SJN5un2N1/+Y9LH933xlz7tv/TQ/Y3lOaZlBpbte3Tfcj3L/s31V+hL57MrymRLJLd5Xt5V7r7n7cH9f/LxbGOWr1bpupxa89ls5tn2LArc1E28+eWlZVvu1dwNQse8vDTTeRBGvjf4R/f58fPlPumnTbrNeP2Lb74WDd4PQeAd3v/0vbX/Hde2fuN9rQ7h5198/+P6p2a8TTfbfL6blVm+/mL88ID190kOPa7/3+Fz1/rPFunsJv6YLG/S7WRz+2AaUv+7B9bf8gLbaa2/bXk22QtfcJwHP//i6z8cDp/lq02yTY1ykRrbdJkmRTo3eNXpapZv59n62viYlQvj6WaTrufZJ+PpxB4byXpuCA6hiklpFGk6V/UKqrhJk3IyGLzbrQvVcLI0fnqxzGc3E1HmJyNfiyfX22SzMPIrcbFJiB3/V2G8yK531CWLyJTiwV/TbX5aLPLS+C5l5kyZYllS3yaD9/R8nhQLIr/M1kQ+W8s+XclWeHB/2RGbp8XYKHJu71bcXFNzNPSyoi66MjGeGs/e/mhc7rLlXD0ZpOsP2TZf8z6RrdOIiCGMd2dPn78+M7LCSNf57npBQ/6xSK7TJ8bmtlzQCFt7yDg/PeWZMizTdi/4Kt+Vm11pTCbfbNNit5TbUJaPReXJz0W+vhjQQg0G2WqTU3eT7TV1u0ir6wUNfpldVpdcofqeF9W3zTIpr/Ltqroudpe012dpUZcobovB1TZf0ZyU3J6h7r+ly5r2z9n658Surta7FQ0qKYz1prpV5tvZQruIr9N8lZbbbFbdvk1WS0mqVWAyT8qkovucvhMLnb393piKTozi+CpbpnF8MqHJypcf0tHJhHmXpNe5fTF49v2bFy+/pcLDYjuL6W5W3n4zy9fEB98wAzH/xOU2WRdCxH1Iv1mTn0EFi3R1uUztCfdrSBRfPX3/8vs3P1BL58NldpMOx8ZwnhXV1ytipfX8z7QnhheDN98/P5MlX+3k4++S9Q/5MuevZyvi53zLX/8jnyf89zkx3OJP7Grx1bNF+vEymc3Eo/+bXF6KL6/SLKGmz55/K5seGPQZ1QSqflSETsatAtDXqgedMjCIqnd3tQPd7hSryjSDqYvAXEBjcqB1IZilw4X0easKiZmqy2iz2S5yMRgM5umVkgqxFYvtPjp5IirT/vqD2PGNTEqk6GKWJFljzJZJUaSFMc+fGGkyWxhXyYy31a4wEuNqmZF0JImY034Q0rK83ZCQMH5n/Ne7/5rw7mUq7GMXvKKCaybZep5+GvHNE4P2png8NmL6x1JGLP+FqFcm2bJTj2/KerGoxNftetQJrie22aQkTs+3o/OawVVDJPUTFqp1Y9UNrSMnosV0fp3GolqnWTE42Y1Cld6m5W67Fjt5JG7oTUxlA7OkHJ03d8dQYsITOzJPLsbGPFtNLbXUdTM8QGxFDHisxv07Y5muR/VoTy6gNkmueJ3P02LKZcS0tp5Wk1BMtWHaxm9bzap6J4q/lnkyj1fU9LLiLRKsE5apNN1Fui1HJqmhcjsSku0bFFbDEzltQjReCWUpGioqkSgU6EAubbpikZ7SMkiZPHmv7oyqlqU8ZGFJPSrTT+VItS9lItVkgTcpkqs05l6PqjapBk3+dqRYfzqU+tii7XS92RXTc5PWI93ks0UxpcFc0hrQn9nNhoquSTlX41DLL3o9kkTPh2JEw4uTSfohoSlSs1YkRDodiYdjsefGajuwvpy+oVbl11jonOmLZFmkan6zK/GE1TCrdC77pF5KuXirZL1LljEXG/F/sn/rzYS0wTxfTeoHVWOSjEFzn4oGZQW5s0WL65wFyLxaYxiuGIO0ckZyIN19cnFSDXyeXadFOZK2lmqMFXq6pfVRmn1SLBLb80eKO2iTyrZ4c8qKTSdk3cluM2dWkOUmYjCkLMv88rZMi5G+PqrKIv2kOlP1bZFvYrKPy6wos1mh+jgWxtRtvT+mUtI2cvQ1mZBsdy3yj9xCIWwtWTedT4y3eZFxPcPiBStupOC8TGfJjuaad/Up2+RlbU1ekcmZlbUMFTfHSubFaq+PjXWyolVTF/OM6InNy0JTTcIV9Ze+8vazT6op2TubUniuGxFAreh7Xk2ffExPiZOudsvlaATSxGi+08WpJSsl1wmJgVLWEVbJHZXmQr5d5vmyWXq5K3i4LZEPPMAbPpvL5/SFiHUUztjo6BJgZDmw83ZDF9RS/VDdG1fk+GFbtXA3m3bV2PfWfL/dpVKybbbphyzf0TLOdls27uipWPTzJ2PDenJqkfCpr+0ncrGYVWE96k52GmuKNytR9etA4Yp3WeMxfyHdrCBF9SEla1xQVQX/9xS5pypIpsnNx2TL7TQMqhozplPDqqTZSBvN7w3zZJKsmVlp+VnCjbrVf888nSyXmjxKMuoW+WAlefhnWzKxRsOrfEeOW8ITIJ0kJTQToyBnaik3YOUQVW6YNIiGJ3J9kmW+ZuXxK0yi1HO02LgIOAruJy8dj7N931JzKeoLboXNLAeoarbuO6qiFEhC1kxbnNBlSV16aYLwb/XUDbmx4RPaW6W2GJMi+2sKhsKwWvK6REwLTXPHfoZgi3nMCvGJwZt4NKo55H/o4kXvtlpJpCN2SpnHYuuozjXdbXUlJjnBjEZlyIogu2j0a8On52L9Loz/afwKPKnunkxWabLWSLfbVs3UNRsi+2j0IvELDFQMkAYqBnzvQKvOdAYMffl1/4B/7T3i7kh/3dt8v9ZxsKTXyRVZJeVsQbsvTlSEJU7shtYIRcwUJdcJ0RxpckV/vIf4sCQxG9fiI+ZRUtMJub8xW51ib8Wf8m1TpuZd7Mj/w8GetAl32ZeMvyKd7djpjjUxEcNG0x5Mit2qbxtyGtk4kw8SMkJvVyTISZrUE6mky7lW9WLPFOkCQnQwJtvh+joFDqjY47yRPRc9pE1rSX6prF+2OEW4pzhkAiu5XkfoyK5R1to9prOorFw7YeIk6+t0ZJ8omSvjdke1phnirLcPN98VrDLiIxrI10ueGo4VMgtUQyPvgmeyviR9365ORrskLxtq2pDDUS2oC62+vCcibEUszWqspi/LKi0TnoERLADfv87K0W+T7fVOBM67HkATW5tsd+vR+ZAqkO/UVCELZvZxPmU/jb4lm5IjEjIUKCaUDC5y19RXwRdynidFOadi9GdLbrHSx/9mcAT0j8SgSxHTTGm73hi79Ya2J4cwSMPPaDUN3uik52c0rWtjt6E20mRFl6SfRDR3I+MbWUlWeb7bzlIBDZPlLX1G7gXNHDuywqpGv3VsDIWJcCojl8ML5WZydSqbFxMVQZ1cp+VoePbvb8/evXx99uZ9/Oz7169fvh+eVKaPqkN91t24ui2eexLOH05FCFSEwc6ePh82RqbW0YkUqaPhHUDH8JANIGkSe8gvmiiap+zjZ1cZUSLLaZbGdWkhLUU32XfaFdzH01Py32ckBbK1vGR20HqqiSDmN8WWugAkD5ruS8cen2x2ZHSwn1/EyQfSoCzQqSA7EzT5BVGaxzT1ydVVtqYFG5kaORmspvJVkHgi78Qs7GmGRiftHUhlpVMbV2XiuLNL67BuVRoCvfvriT1Npck16hSoNyXNYWXlCh5gR7kKiU+eqi32VjwZzdNiRltF+KlxPM9n1BrLqhW7gttYxPSmdeV3ycfnTYXv0uXmRVX0BOhNkvk8rvbyaChD+kPpiU5JkY1ZSCS7ZTnlMP+dNeWeVxxID3hnqaLiDxcuqo0uZDK7ohBhkjKJo+bTTmSzDjm2/D8ZuK24nvUYFbhH7nM/JvuFv2pHoBfUEGygSn7SitaitG4IeYrFRiy9cBbKG8G7WR0WIcE8Ebe7ljeZTVR2b6CivaFk0XtUbbt7vyiXlO0TBlYmcxo4UxDDHRs8o+tyap/UEky0INe1EV8CvoAnJ5OP26xMZVTuznaN3xnD/2QZNRhQ43HMLlAcs3obxjHvhTgeSjpyYxyXnnMX/ltdpDGHKWJGTx4EA9+D/7q+67fxX8cKnEf89+/wGQ6H73ZrHfuVOl3q0U1OjA/oq0B9FWBM6ljEQ9+zsjEiDcMVQGgqAZTa1FDh5LESZNxWSSantE5269mCTUfagzLKil1SAWs2MgX0IpHawU+np6vk0ykbQ8u0+MkoaNunDMfIqC7bYYLex0VGlcim2BXp1W4pY37ZipHty10p4h9yUGIkOKga0JViQIK6+zeGcXqqBlgYf35jhe/evVwrnPHJB4ufph8yNqo2O7q4ZJ/vlKMJhmX8Zy0nDgDCDR0B7j4QDF7m19c05upSTlJ9lUqjD9HgCsUlAXE/MnwQCxY1EQivqlc64WtAwd+dPfvj2+9fvnkvgpISaGk4uvhGoCvKfJ1sysVw8PLN8x+fvX/5p7O4F46crWsQ2fJrDNnyFYj8/t3TNz8c1eD9wPT7p394dRZH8ZunrwUuLDXt8MUfLO/GdgJkN1J0dP+Uvkg1NuxyJBf58xso8ubs1at2gTevoECVesEP6u/jwS+D99+/OqPxPjujTpkT01KWWrXxvwAOVjECef/LPhBZXYT8r+oJy5oBmvrc2FhWGKunjTeekvlJ2532C3p/bbjoII4kH9wFPinUcjdPsEDMoRNZCOBEySsj2WEZ+uftEbNWYmuTBcvYEDIlFhHKJ7VdKAQgGb3KmlbBVd4rpYBhRsMnw8pEJbk3NfZxLlkeFQzOhke9+BIfa++cCmNk40Y3CFUjxEBV15pnjCwyZ5GxNxQEpbhkejxJiha5CMvlEKpJGJIqmnDzUliQnbKMULIJSEzXCAjN0CO6aqLQ9RQjOR+qJ0OGK9R3OdQrDofz4kxwtRRf87SOq0am6u9JVXGiJmIi8ommom/10jalSDhk6wnzaNysMhVvLpCxqcZY0K1THpbkwnCEUXoG4nnNHKJVvjqcCkGq/Wadf1yLnIfCSD/Rn6UQ9j/RBiMzdE1C9PYnY56T+hXadZEWdepEUWN37ODzaJLi5nzIfDgUgZphLXZerq9SEuKzdNgJquhJBPzZn0hQD2eCKQXNXbL3yBOOm4d7cgvq1jv5BXrj/PhQ2/zsotVmk3XQ1KnvNUX7JU9UnHNgWPsyJQ5XESMZdPuparAH1upojdyTkSV9x3pMZOEln2JllimuWlfXjKIdXgzp551bFxW/QEtVVIhhMLwt0DZ10WGbmpZsUMtcYaFNxupqVFc/OX8CLasQqbAitW6LoJlysOXj7kjuHifDV5L7DjUiEogOtcEPz6F+E3ikZtTCCEM8/pAsd2khgVg5OSWbtrwMwu9kiVXUKSNanEw2kKyT5W2RVZeytrBDh3tSS7b5R1YxfIe/sizgv5w0KSqeDyPa87SudPd8OC+kANAsm3Puq55A9LfharvlEDFXejGkHTtcZGVhmfW9xfCiChBxXsmOkx+komSvW7gcu4YZpSKVNqAKAghPZFqVE6XYXhbRJWU4c/Dylbg3Gqam0pptY6EJIFTCe6zcl1gp2EMaXdQEtS6vlW4XF6DgVSxaqkipe5g55B0W8E1kSDKvKCKSWmN1U6kBUWMq/yjLpGI5BqrV/rcvJmU+wlJCIcRSIUx7q5h2K9ViiRDUPkkiBo7iRI+FwRrKcE5MlKesZMTVSVNY6maOEjEANitHUgaIxKfGGmDRtYmpUZXbQt7akBzINLtelIVALFTqUZPIpPVHkmoNsooKtU2adpaSMAbJn6GF3KQJDStd5dtb0ec6CChyY1guZGzFpdsrYqUd2RTbUZ8UJelbVTJYrpLaAPVKNAal3ABjtbip5JwpLPxYbdBYtQsBwSIlFhfJjnt6apzKcQw6qqHuxX2qgdNtYrbup1j/CN2ykmlKzHiVeJHYnhrLubhJGhxkTasA3/8/9ODiRIUJWUKKCG5b8h5AGBp7WIiAPaF+EB2teGkRc8S+5nSRcMZhWbSNlQMZb7SSYorkTivQaK6lC5VuyRuM7XNtNeE0r3Ezhj5Tf6ihet15Lqq7ULhabl4E9RUBDKmp+H6VU8FoBt+FUtWKi3C0/IpPtwx6X9X2Z11GMQLxrGhQXeJqZCz64zJfpmRRiLrJZTG6u/6J8XvSe7XjTCKi5ug66VALYIvtJALY4lv3WSzUs5i5phjp9H3TKcSLwigrMXN9KUBYGQZibbLdcehrVN969+MPT789i384e/WCNP+OzP5P24IpMMzx29/a7fbJo9Pa1jzeT9UjWjWWtuQbn9RNOQdFZnty/olRocr7GTZ40N5Yzbgbmxl3YjF3UxKTNGSoOKcvxfRc6S05ZxfQAXHjzsaayOR+SOvOyhCOhdpfCTzjCT5lW6mq/la42lX9fSYtV/nm5vruGVCgHDTaG5/bSS3WRL6U13GVs23QxcDqx5NNvhlVYr+5fQ5Byypd4aKbGNzYL9Iir7J8tZzepk3FLxe1IyT2ZGM/CpBpZJ7cswsFk7XRPwT9mBoxIck0FlznF1JLsk/ABNgpqG1eESYoNkuGzcdDLaVCNH0uW7moAP07jfwmGeAQcFc1dy6ySWuwjX2CXbEAM0ZM2gFQr+IJ9eCfBNI76nOHu/fFaDzk/K/rOI/nf/8On2Pc/YfSuBv/paV2O+d/beKBR/z37/D522AYsWge0BdhXTJqZHvD8WAohDHfqe0QgwTtlh/9yOH2iROG/H0hLnyXvr+X9yOHv6v7Hn1/Ib66Jpd5oe777uCXsUbWM/uRdYCs5zoNXcdt6HpuUBN2PLsh7Pl+m3DQb7yu6cB4zWbArmnBgC2rGbEVwohtu03YMvsN2XUjnGm7oez4QNmBucYh+76jUf7zH3svseXDXDsNYcf0GsJu5NeEbRixG1ltuj3X2PKBrmM3Uy0fKLqO5dR0LTETirAdhG3CPdfYEfxaDzhoBuyHwFxA2AmRuZwO4b5rbIkdpCjbod8MOQTKdhQ1Qw6DhrJj6vvpzavTnttJsFM1Yhu2UwhL7LkRjNgC3jL9Nt2+vBXhVPtRQzgIgHAIhF0Ppjpq0+3JW64ZAN1mhaV4qscbNmR9YC3P99p0e7OWJreAtTwQH57dbGLHgon27M4C9xYfgQXiQ1KTD0IUH0Gzi3XxEemC6/V7yyiTT71I2zZwtRIaatDA1YqcYn0g7YROh/QiTZZ8aL0P9Qim3AlARQU+UgdVEQKLuaHVom4b+fa6F2kTudvyGu42I1hty2vmHNba8tvDto1ilvUbs4dj9mDMNqhl10c2Q5nthS3SDvk4ZU/SoKZcs9EWdghby3UbBemYMGq3pZmZNHmQ26QXcd8C4oHXcJrvgRwLomat/RDskdBur7Xbf8oDYHKpq9W4I5zxQ6Nur7V7DIv7uMECUFg+SlFyv2DYDfGWnnzNGeLJqt+gHc36Az5zgcFDC2ySAORo5Hgta4hDXv3EWQhby/fAGgrQ8Gzm2zNBpARtI0zkN+2lCiRRkPheo5w9C5jLDxsxBmLEj7o07ftpehYo5mZdpYKo1rWxQzwHSBIjdEg6fYapKYpmTT3NwvUbleyhivLD9pqefnB7UA3B2FNsq7qD6xkGsKA4VlNXT1VO2z1UfReWNJDWnVou2LVB1AgrP4KdE5pOh2qPRSVTHKmClApAHwYhUoVdE0Qdoj2WVWq7Sj4FDfdK/6Q2ecBx8cG09M2wQ7XHskpHqBoqqH4589VQ7YaFfc+BododA6/PsgYoHCKrMWdDE1R+GIVAFZe1JY2Zap+9avs4VtitPjKT31D1QhTDLQnBVHusqxQ+NdVGEHpoXAUuCIkIZX93qH12a4B61gKiDhIF+8LzcLeGupL/7rVh3dxP1UTTwowaqmZgN1Qtu5lg04exWlbUpur0oeqAq2CCZDJxWS0rAqoBUg3bVL1eVF2kagFVHKsFRF0k2plgopJd5z2WFuiqTSrva1EGFzS6DRvWb/m+L1+9fWb8sEpECuZ9EsoEXediTMdEo9UDo9WE5XVbzqAg/Yox6PtJ22izQqTBMcF+coGxdHO5NWg2YqJIdwjfQ2I3uqEYWoEhu8jRvt3wlhth/MzzOoTtFk/vJ2w7mpXeLLNmrYKdbDuwyk7UUgbP+b0R6bYHYRkpqQhHDWFpRFQjNmHEmv+rE36Wr2fppnyTluQG9xm3ieQtvxEhFpoZDrC37WqOkdOi//zs0x7+PrDaARD3/eDAajeT7uLYybzeQ7vL4AfmHS11FwMtDgbxfAy0gMHstiK1//H02+8dy+wzaA/NSauxAFwTWM23YdQWsnigE366zYoy/5Pbg7KFay0jClUwDVhNRtkqeYJr7epr/fwPm3SeJT0ZzUFnwUNzy0Y5ajZqGeNaNB26/G4OWty7sz1QV04AfmiAWxtiLS7Ylk5gtax3y7s5tZ1gZJknfaijQe3A9pZUqv0VOfvJu2Y7PF6Rt/uRxx2mNJZ8gLoabCE7cHDF23BERd3rRd1BN1xag1XURQvuQejDAgXmsaUwuKAbIs3sgXAMwhMOhNoO4jHIdl1YpDceg0N3mnW/A48BI8nreKp94RigCtx2GIyxcbhOm2p/LEazkkBv61gM7G4fp7kDEPRdXtvCaYblRa0N7I3eqteK0h+BxGioRCNHD+IwNqyt3aHaN1aOEhSswUMgjMbKntch2x+ECXG4x4MwHTCk5x5CK9SDCP1BEEaHFq2Or9yTq3QwJAK5dQiE0aWG3RlwXxTGwiWO0DbAEaPFH6E16Nttwn1ZC4O1Xh8UxgWbxHMfjsJgYF7GnisUBk1ACDjJwFjlOFs65WNQGAc0gwtYhBPhIpuNj+NaqBtb0YkjQRjd+oQJP4jBoJ9DIu3BGIxlwlJb4L1HsJmlJ6A2s4nee9Ce8P4gjIOxLxf3s6NxGdifGshoey3S/UEYx9SCfY3wtEO0gxCEwbXuAG5HYTBogIYmghHA4hDcDNCLD5029NQfgnEwN8MNwAjREhUgQC8ttlo/Rh3SR7B4gIwmI5Dv1QMYt3ygBo5x81ZU9xgUJorqGbcmZk03aja2vP1C3a79Orr9cPwF3Tof4GSMYfsguj0HR2t1Mn/6BHY9E3NgIMJ6EIFxMdhpt52bnnFdXNhmpAdBGC2C7XUH2ieqi8wk8YL7QBhMuvGjjiHfK6wbIQiD2ukgCINr+kAMBoP1QdQMNcBdG0CwPrDA+ggtq0O1DwajRbDBTfZDbbeClEL8MPI6RPtAa6gRPBeDApqFBzatFu5rRQV6YzDIv00YxMfptZs94/sYqnettq3TC4HRlACI4hATbEKwoQMf5jdqxfn6IjABshIIJS+AXRPISLmaG6Aadi33XntVg/Mg6UCn6jVj9U0fqT4MgvGRgR2QEGjDBgHIwkAbq76ufSEYTBQDUOIgABNgRksretobgGlImpBD0w9+6dLsBb/YwEkmAMNm4CJVhJpwpB2iveEXB2PETjPcw/iL5gQ6OuXj8BfNQ+iFv2CiVEvTHYO/yMBrHbcDUxl4Wd6vDDcg3BZRveEX3ZyAhA5Tm2sAbNGG8dtZgP3RF4wwuBDFQbqecyChtkX2CPDFtdDRt3HAGtzU8Bbmk3huG/w4EnxxUPvZ4IFiiBCEs56y3UZ+jkJecJ0houNiTpqW++gjqOnuo90beXEPeSWYAuGCQ9RCXnSl3x958VAHo92KuVnKrFEVDqZIHQG76OPFaJKrpYtjLjFMtmd/DuwipWSdng+BWQ+iHD5oCxeTXFq0j4BdnAiDdxgAj5C7HeBuhLnam/po3MXD7EOrWWzbQ9kN0R0bYZfPRV201MdGntmYi4fRBhuzLtvA/fGwC+aQQ3zasUGqeWDhOhZiD1YbfFDkNdJvt6l4/RIfU60Ja8ncePgFA+Oeh5gqcrrT9pf4BPC9VBFhk6Z95QOjLAPMAzNv/FZYXIqy17TDdqv7x4vomvJ+1XjRfUF8C/1Dl51SiW/xa+UehG9J37Oe8qYLoeYs2s02CyN0Fp02q/UNVfvoGks/Q3mpGO4AoYapXWHn0E/POLWvocjgWoTI2gHAapEJHBZ2AM3egWrEEQPYUxFCmWEjTiLNfeuegOm5vqGmsxrdESJkH0K6YORpbmPUpttzeUP/QBAiRLQpBHtfTkQ10a0V7g9yRRgSCIGvIgtTFUPnwEy3VvgImCvErRRCDl+ESU8hhJgiNIoip5Ou2HMnoYfjmRiP0IA9CA2gpvSCDvjSW3Zo4rohHLho/VqN7AhsiB76Tgdt6is7NLqN3RvgoQQfzP0QkZegGwvpyVsBHj/xIV8yCA8JLUcLXXbCIX15K0AbEDVjiAZ3AEcGIwdBgLAdie8Pc0Va7i+IkCiA2Y482MkIkkefh3MFGkAPEagITaEIMNwIEc2og7z0B7oitMIigJsiPOAVQaC6RfrhSFeEueURjtp3cdQWkPZwzttny/ojXSHiEAGkgIQImocA5UYI5ZKs+wyoS4riatyQjoE2f2QC7TDCGX8w0hUhmBtGOOG41m4jyiR3VJS99qiPQrpCLdHdagRa5KDmiHCL3cHjRyBdFkIxDdJl43E6E9N9gK5tt9N9eoNdms3ZCLNAT+VrllkXo3uO4fQ6moIGp30AFvFQeKOaDDqj7QeLaOdxG4AtsBwk2tBE9znwu/haj/i5BsV4EHfU9TIws5YSEXZo9sFEtISEsBEbAW4gHxLlQswUDKLuGZweaxpidAJOogaRZoCAhEZ8OvS6R3B6LGnooSkP1gfKZg9wAp1o+7UB/ZCuQD9fDYyEwScPzsJI2VEbAA86bYRnyEII/rSGeohmF/3ps09xTdWBZvkAI04ynF5xL4bZnC6+1mNRA8Rh8Pij7o6CSRmhX9g+N9AP6PIRcvJcFIRoyFoNK4WI1O4bap+tquWyhMDAqGUxZSrE061hS8Mfg4hI1V0xFKR5RhZgTyHuHQy2RK18g2MQkVCDMqNm1CGOOoSTIREeVIjMzqmjPvCej5rdhdcTqBQE9QBcJB9TH702zT7wnhdpOduAxOOudcH11v1Bt3Oqqw/A5+PSurCBtCwlPPITYAqvFz0Y4fPRYvUgES/0MKYDqify8My0/WC0C2M6LpwS91BoYBKedjjee/Bho5bcAGXrobIFwqGWK9RK8zwC8PL11200zqB+vApEh61Zbp952sjCpTbDRn5gvBTP3ASRFqvdAzv1RbxgxuHki/LN0GGpHBMMKgVdun3RrhB3cwhH6SKEGEMHlDAaVq28mv5gl2Yuu6j7Q81yBYMDUxOiFrZ5BN6lQS+2iWnboCRsPAsbgcx0W6lax+Fd2puKIDkisLTMbVBQeOIkbBnsR+BdvnbgA9Y5xGNd0gys9BMQPoT49EO79EgexOIjBAHAD4wwRyIM2slxx8Fduo8EFl+IQHaAESY9SN32zI6DuwINxAffP8Rc6hCssAjtocitURj+feEHojC4+JCdeBCFibQc3+5xn6+HwmCU/qEwTIBb3AdwV4NhkG6Eqb1hNwe0d5AeT08GEDs+jMOgAep08JC+OEyE4ICDYhzDahC/1XjM66ASn43DaAAQMBYe6Grv7C8Bw2BIC3wMfZ7dB7/xLcKIVgjueQv/AVUdjPXN/IVBGIwuodGtpT6ZHSykJ2MF+DYF3wbHVTsObDacFeBBQVI7DwRhQkwxC+DFjQFm9PkAtLXiaR3n9UEoDOT/4HlcPwAbFB25oAt39ZYd2qsqgwOh8QCOC7biIV0kpDcIA+ZnBHnkESpoFJdagLgTGD8KgonwGF0IhwTugGAwNh62qR8BwaC9H4FejEKUIkHYn3R/CCZA0oAIAJdFKDi/FAITYagvBNwpxEGHNppDQNn6nPe9RfiK0MjFtYZRw4sFI3yRUtQ5VnYEAoMmQQTJjBG6lhppzHKL/C74cwyPo1kQIQKjY33RoeVuvefuCAhGS3qyIG3ARqfWRNtP01jRg1/5pmUfAbCrh+wdMP4CNDrNLjjRJ7yLtr7VB4XRXnTUPVDf73AKUnV7wDAYhSPT/2EwDPoV8Hq7wzCMNr/d6e0T3MXI0KElRRRGe12j2T0P0weFwcP7gYeWAKbtobGHyZLhnvev9YFh0GEOfBwrclKApi0uqv+gt74dxmE8baxAVRPO3bNVvWAYDZPHJD00tTyYYMyLDKPu4Z8+R47w3Z8eGHgeaEAHchBCFMjtndoTh2kFGCGYi68zwkXVHMMHvfFND3WBdRVoOc4uIk7ae+a64E+PRcXEPBfEvp6JiCFF7ZCTu+flXH1RGBtDipjRcxiFQVVr7Xkl2ddDYXTSHUTk66AwASIEXjuq99VwGMyOa6fOfz0YBjGJqDPUvjBMgJOMedQhvhArgDf2RggARe1Te71hGDwtiBbFQRQmwGxAr5WmdQQKowVN4TyInoYIYcvQ1+ynBx87CrSfTAAbCmNY+AawEF/AHbY20dEojASk9+V6WHg+wLMO0d/zzrfeMAzi9GCm63lx4IFjvDLyPwOIabllDWkTE5pdjCt9CRwmOpCYFmq5UvBeZj2cpVuuXwKFcfE9Hw6+UxfPJpCR/TVQGIxKI6CLKGPbsjsChNGDaQBvRqiUQzjQF+HRsrDzTuqvDcNomZeHTjx9LgqDb5oLMNaC7/iNnEMvuuuJwtjaMRw4IIGHZUMEgTRGb2et9T1zpEUTAxMBAbRIEF5FreW1PaZeR44iSwtSg+pAcNWHw206nuzvAbJ7njmK0KoPIcDjaPgHyhMt0dVmsGvwyz/6J5b+qT/4+1/b9C+7bJuKi9PrzW5SfrrzJ6/6fu7+/S/Tsjyv/ftvgcfPvwTx+z7/4r//9W/Gt29/NNL1h2ybr3nlxa8zlguylxvOGIsflc53pfH29j3/XuRY/mzkaTETv4JqJOu5qMM/Qcy/F/1xkc0WxjpN58azH58/PS026Sy7ymaG+P3xYjKYpduSbkyntmn7E8sitT2YLfgHNcvTNf+46jL7a7qdTp2JN3EGs2W+m2+y2c0y5VsW3UqT4pZ/xHs6tSaWO7jKlin/EPh06k7sCV0XTFI1H03MQTZfJ1yX6PycrX9ObNmQP/g5v1xml9yMT8VWyfZmtymSK0GIjIbBarNKygU/p+vBOtl+XCTLglqe2NwuKeuP+fbmExcPBuvdanPLZYmqO9gks5uEfzeeSvvU1Kbg32CfTgPqoj3Y3F5m67llyY6YdM0/KCpKO0SLit/eJisqTv2ixnhzpkUpKDs0yEExy26y8nSZJts1k4wmFt+T9K2Ar9JytynznLsbhIJIcas6aLl0VS74x0I3VGJWLplsyDfFyl6nufyVa6Ynyv5lvuLZDUxqubzdUEdP009lui74x5b5icUTuNsuaTodrsWNfVykKbXMtjdd/aOZ/fHz+Hn8PH4eP4+fx8/j5/Hz+Hn8PH4eP4+fx8/j5/Hz+PkX+/w3U5GiWgDIAAA="
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as archive:
    archive.extractall(REPO)
print("code ready at", REPO)

In [ ]:
sh("pip install -q uv")
sh(f"uv venv -q --python 3.12 {WORK / 'env'}")
sh(f"uv pip install -q --python {PYTHON} --index-url https://download.pytorch.org/whl/cu126 torch==2.8.0")
sh(f"uv pip install -q --python {PYTHON} -r {REPO}/experiments/requirements-gpu.txt")
sh(
    f"uv pip install -q --python {PYTHON} --only-binary :all: torch-scatter==2.1.2 "
    "-f https://data.pyg.org/whl/torch-2.8.0+cu126.html"
)
sh(f"uv pip install -q --python {PYTHON} --no-deps .", cwd=REPO / "graph-walker")
sh(f"{PYTHON} -c 'import torch, graph_walker; print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=WORK)
sh(f"uv pip freeze --python {PYTHON} > {OUTPUT}/e0_gpu_freeze.txt")

In [ ]:
sh(
    f"{PYTHON} check_walker.py --output {OUTPUT}/e0_walker_check_kaggle.json "
    f"2>&1 | tee {OUTPUT}/e0_walker_check_kaggle.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 8 --data-root {WORK}/data "
    "--datasets FB15k237Inductive:v1,WN18RRInductive:v1,NELLInductive:v1 "
    f"--output {OUTPUT}/e0_zero_shot.json 2>&1 | tee {OUTPUT}/e0_zero_shot.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 1 --max-triples 20 --data-root {WORK}/data "
    f"--datasets Hetionet --output {OUTPUT}/e0_hetionet_timing.json 2>&1 | tee {OUTPUT}/e0_hetionet_timing.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
for name in ("e0_zero_shot.json", "e0_hetionet_timing.json"):
    path = OUTPUT / name
    if not path.exists():
        print(name, "is missing; check its log")
        continue
    for run in json.loads(path.read_text())["runs"]:
        print(
            run["dataset"], "| MRR", round(run["measured"]["mrr"], 3), "| paper", run["paper_all_relations"]["mrr"],
            "| full run", run["full_run"], "| s/triple", round(run["seconds_per_triple"], 2),
            "| peak GPU GB", round(run["peak_gpu_memory_gb"], 2),
        )